# Round 5: Linear Regression with Feature Engineering

In [1]:
## ADD SAVE PARAMETERS
save = True
#save_name = "round5_linreg_lasso_fe-trees"
save_name = "round5_linreg_lasso_fe-linear"

In [2]:
# import statements
import numpy as np
import pandas as pd
import json, os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import sys, os

In [3]:
# load processed data
X_train = pd.read_csv("../data/processed/feature_engineering/X_train_fe_linear.csv")
#X_train = pd.read_csv("../data/processed/feature_engineering/X_train_fe_trees.csv")
X_test = pd.read_csv("../data/processed/feature_engineering/X_test_fe_linear.csv")
#X_test = pd.read_csv("../data/processed/feature_engineering/X_test_fe_trees.csv")
Y_train = pd.read_csv("../data/processed/Y_train.csv")

In [4]:
# Linear regression is sensitive to feature scale
# RF and LightGBM are NOT — but linear models ARE
scaler  = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)      # ← use same scaler, don't refit!

In [5]:
models = {
    'LinearRegression' : LinearRegression(),
    'Ridge (L2)'       : Ridge(alpha=1.0),
    'Lasso (L1)'       : Lasso(alpha=0.001),
    'ElasticNet'       : ElasticNet(alpha=0.001, l1_ratio=0.5),
}

results = {}

for name, model in models.items():
    cv_scores = cross_val_score(
        model, X_train_scaled, Y_train,
        cv=5,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    results[name] = {
        'mean' : -cv_scores.mean(),
        'std'  : cv_scores.std(),
        'scores': (-cv_scores).tolist()
    }
    print(f"{name:<25} RMSE: {-cv_scores.mean():.5f} ± {cv_scores.std():.5f}")

LinearRegression          RMSE: 0.13424 ± 0.01486
Ridge (L2)                RMSE: 0.13422 ± 0.01461
Lasso (L1)                RMSE: 0.13352 ± 0.01496
ElasticNet                RMSE: 0.13392 ± 0.01493


In [6]:
# save to output

# Ridge is almost always the best linear model for this kind of data
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, Y_train)

# Lasso beats it in this task
lasso = Lasso(alpha=0.001)
lasso.fit(X_train_scaled, Y_train)

preds = np.expm1(lasso.predict(X_test_scaled))

submission = pd.DataFrame({
    'Id'       : pd.read_csv('../data/raw/test.csv')['Id'],
    'SalePrice': preds
})

if save:
    submission.to_csv(f'../data/output/{save_name}.csv', index=False)
    print(submission.head())

     Id      SalePrice
0  1461  113325.640293
1  1462  157894.137102
2  1463  177969.195937
3  1464  199007.596713
4  1465  186611.135824


In [7]:
# get cv_scores

cv_scores = cross_val_score(
    lasso, X_train, Y_train,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

print(f"CV RMSE scores : {-cv_scores}")
print(f"Mean CV RMSE   : {-cv_scores.mean():.4f}")
print(f"Std CV RMSE    : {cv_scores.std():.4f}")

CV RMSE scores : [0.11818927 0.14268864 0.13357821 0.1199452  0.15781319]
Mean CV RMSE   : 0.1344
Std CV RMSE    : 0.0148


In [8]:
# save

log_entry = {
    "model"       : save_name,
    "cv_rmse_mean": round(float(-cv_scores.mean()), 5),
    "cv_rmse_std" : round(float(cv_scores.std()), 5),
    "cv_scores"   : [round(float(-s), 5) for s in cv_scores],
    "params"      : {},          # if using optuna, else {}
    "submission"  : f"{save_name}.csv",
    "notes"       : ""
}

if save: 
    os.makedirs('../data/output', exist_ok=True)
    log_path = '../data/output/cv_scores.json'
    # Load existing log or start fresh
    if os.path.exists(log_path):
        with open(log_path, 'r') as f:
            log = json.load(f)
    else:
        log = []

    log.append(log_entry)

    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2)

    print(f"Logged CV score: {log_entry['cv_rmse_mean']:.5f} ± {log_entry['cv_rmse_std']:.5f}")
else: 
    print(f"Not logged, change to save boolean to log: {log_entry['cv_rmse_mean']:.5f} ± {log_entry['cv_rmse_std']:.5f}")

Logged CV score: 0.13444 ± 0.01476


In [9]:
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

kf  = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(X_train))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr,  X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    Y_tr,  Y_val = Y_train.iloc[tr_idx], Y_train.iloc[val_idx]

    lasso.fit(X_tr, Y_tr)
    oof[val_idx] = lasso.predict(X_val)

    fold_rmse = root_mean_squared_error(Y_val, oof[val_idx])
    print(f"  Fold {fold+1} RMSE: {fold_rmse:.5f}")

oof_rmse = root_mean_squared_error(Y_train, oof)
print(f"\n✅ OOF RMSE : {oof_rmse:.5f}")

# ── Save ─────────────────────────────────────────────────────
if save: 
    np.save(f'../data/output/oof/oof_{save_name}.npy', oof)
    print(f"✅ OOF saved to ../data/output/oof/oof_{save_name}.npy")

  Fold 1 RMSE: 0.13663
  Fold 2 RMSE: 0.12568
  Fold 3 RMSE: 0.17759
  Fold 4 RMSE: 0.13002
  Fold 5 RMSE: 0.11892

✅ OOF RMSE : 0.13932
✅ OOF saved to ../data/output/oof/oof_round5_linreg_lasso_fe-linear.npy


c:\Users\Chyi\anaconda3\envs\kaggle\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.984159e+00, tolerance: 1.781e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\Chyi\anaconda3\envs\kaggle\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.423988e+00, tolerance: 1.871e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\Chyi\anaconda3\envs\kaggle\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularis